In [6]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier


BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


source1 = pd.read_csv(
    DATA_DIR / "sample1.tsv",
    sep="\t"
)

source2 = pd.read_csv(
    DATA_DIR / "sample2.tsv",
    sep="\t"
)

source3 = pd.read_csv(
    DATA_DIR / "sample3.tsv",
    sep="\t"
)

ground_truth = pd.read_csv(
    DATA_DIR / "growth_truth.tsv",
    sep="\t"
)

candidate_pairs = pd.read_csv(
    DATA_DIR / "candidate_pairs.tsv",
    sep="\t"
)


source23 = pd.concat(
    [source2, source3],
    ignore_index=True
)


source1_lookup = source1.set_index(
    "entity_id"
).to_dict("index")

source23_lookup = source23.set_index(
    "entity_id"
).to_dict("index")


pairs = []

for _, row in candidate_pairs.iterrows():

    s1_id = str(row["source1_entity_id"]).strip()

    candidate_value = row["candidate_entity_ids"]

    if pd.isna(candidate_value):
        continue

    for candidate_id in str(candidate_value).split(","):

        candidate_id = candidate_id.strip()

        if not candidate_id:
            continue

        if s1_id not in source1_lookup:
            continue

        if candidate_id not in source23_lookup:
            continue

        pairs.append({
            "source1_entity_id": s1_id,
            "candidate_entity_id": candidate_id
        })


pairs_df = pd.DataFrame(pairs)


if pairs_df.empty:
    raise ValueError("No valid candidate pairs were found.")


pairs_df["s1_name"] = pairs_df[
    "source1_entity_id"
].map(
    lambda x: source1_lookup[x].get("business_name", "")
)

pairs_df["s1_address"] = pairs_df[
    "source1_entity_id"
].map(
    lambda x: source1_lookup[x].get("business_address", "")
)

pairs_df["s1_country"] = pairs_df[
    "source1_entity_id"
].map(
    lambda x: source1_lookup[x].get("country", "")
)

pairs_df["candidate_name"] = pairs_df[
    "candidate_entity_id"
].map(
    lambda x: source23_lookup[x].get("business_name", "")
)

pairs_df["candidate_address"] = pairs_df[
    "candidate_entity_id"
].map(
    lambda x: source23_lookup[x].get("business_address", "")
)

pairs_df["candidate_country"] = pairs_df[
    "candidate_entity_id"
].map(
    lambda x: source23_lookup[x].get("country", "")
)


pairs_df["s1_name"] = (
    pairs_df["s1_name"]
    .fillna("")
    .astype(str)
)

pairs_df["candidate_name"] = (
    pairs_df["candidate_name"]
    .fillna("")
    .astype(str)
)

pairs_df["s1_address"] = (
    pairs_df["s1_address"]
    .fillna("")
    .astype(str)
)

pairs_df["candidate_address"] = (
    pairs_df["candidate_address"]
    .fillna("")
    .astype(str)

)

pairs_df["s1_country"] = (
    pairs_df["s1_country"]
    .fillna("")
    .astype(str)
)

pairs_df["candidate_country"] = (
    pairs_df["candidate_country"]
    .fillna("")
    .astype(str)
)


name_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    lowercase=True
)


name_vectorizer.fit(
    pd.concat(
        [
            pairs_df["s1_name"],
            pairs_df["candidate_name"]
        ],
        ignore_index=True
    )
)


s1_name_vectors = name_vectorizer.transform(
    pairs_df["s1_name"]
)

candidate_name_vectors = name_vectorizer.transform(
    pairs_df["candidate_name"]
)


pairs_df["name_similarity"] = np.asarray(
    s1_name_vectors.multiply(
        candidate_name_vectors
    ).sum(axis=1)
).ravel()


address_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    lowercase=True
)


address_vectorizer.fit(
    pd.concat(
        [
            pairs_df["s1_address"],
            pairs_df["candidate_address"]
        ],
        ignore_index=True
    )
)


s1_address_vectors = address_vectorizer.transform(
    pairs_df["s1_address"]
)

candidate_address_vectors = address_vectorizer.transform(
    pairs_df["candidate_address"]
)


pairs_df["address_similarity"] = np.asarray(
    s1_address_vectors.multiply(
        candidate_address_vectors
    ).sum(axis=1)
).ravel()


pairs_df["country_match"] = (
    pairs_df["s1_country"].str.strip().str.lower()
    ==
    pairs_df["candidate_country"].str.strip().str.lower()
).astype(int)


truth_lookup = {}


for _, row in ground_truth.iterrows():

    s1_id = str(
        row["source1_entity_id"]
    ).strip()

    value = row["matched_entity_ids"]

    if pd.isna(value) or str(value).strip() == "":
        truth_lookup[s1_id] = set()
    else:
        truth_lookup[s1_id] = {
            item.strip()
            for item in str(value).split(",")
            if item.strip()
        }


pairs_df["label"] = pairs_df.apply(
    lambda row: int(
        row["candidate_entity_id"]
        in truth_lookup.get(
            row["source1_entity_id"],
            set()
        )
    ),
    axis=1
)


features = [
    "name_similarity",
    "address_similarity",
    "country_match"
]


X = pairs_df[features]
y = pairs_df["label"]


if y.nunique() < 2:
    raise ValueError(
        "The candidate data must contain both positive and negative pairs."
    )


model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)


model.fit(X, y)


pairs_df["match_probability"] = model.predict_proba(
    X
)[:, 1]


def f05_score(true_set, predicted_set):

    if len(true_set) == 0 and len(predicted_set) == 0:
        return 1.0

    if len(predicted_set) == 0:
        return 0.0

    true_positive = len(
        true_set.intersection(predicted_set)
    )

    precision = (
        true_positive /
        len(predicted_set)
    )

    if len(true_set) == 0:
        recall = 0.0
    else:
        recall = (
            true_positive /
            len(true_set)
        )

    if precision == 0 or recall == 0:
        return 0.0

    return (
        1.25 * precision * recall
    ) / (
        0.25 * precision + recall
    )


def evaluate_threshold(threshold):

    selected = pairs_df[
        pairs_df["match_probability"] >= threshold
    ]

    prediction_lookup = {}

    for s1_id, group in selected.groupby(
        "source1_entity_id"
    ):

        prediction_lookup[s1_id] = set(
            group["candidate_entity_id"]
        )

    scores = []

    for s1_id in source1["entity_id"]:

        s1_id = str(s1_id).strip()

        true_set = truth_lookup.get(
            s1_id,
            set()
        )

        predicted_set = prediction_lookup.get(
            s1_id,
            set()
        )

        scores.append(
            f05_score(
                true_set,
                predicted_set
            )
        )

    return float(
        np.mean(scores)
    )


thresholds = np.arange(
    0.30,
    0.96,
    0.05
)


threshold_results = []


for threshold in thresholds:

    score = evaluate_threshold(
        threshold
    )

    threshold_results.append({
        "threshold": round(
            float(threshold),
            2
        ),
        "macro_f05": score
    })


threshold_df = pd.DataFrame(
    threshold_results
)


best_index = threshold_df[
    "macro_f05"
].idxmax()


best_row = threshold_df.loc[
    best_index
]


BEST_THRESHOLD = float(
    best_row["threshold"]
)


pairs_df["predicted_match"] = (
    pairs_df["match_probability"]
    >= BEST_THRESHOLD
).astype(int)


predicted_pairs = pairs_df[
    pairs_df["predicted_match"] == 1
]


prediction_lookup = {}


for s1_id, group in predicted_pairs.groupby(
    "source1_entity_id"
):

    prediction_lookup[s1_id] = list(
        dict.fromkeys(
            group["candidate_entity_id"]
        )
    )


results = []


for s1_id in source1["entity_id"]:

    s1_id = str(s1_id).strip()

    matches = prediction_lookup.get(
        s1_id,
        []
    )

    matches = list(
        dict.fromkeys(matches)
    )

    results.append({
        "source1_entity_id": s1_id,
        "matched_entity_ids": ",".join(matches)
    })


matching_results = pd.DataFrame(
    results
)


matching_results.to_csv(
    OUTPUT_DIR / "matching_results.tsv",
    sep="\t",
    index=False
)


evaluation_rows = []


for _, row in matching_results.iterrows():

    s1_id = str(
        row["source1_entity_id"]
    ).strip()

    value = row["matched_entity_ids"]

    if pd.isna(value) or str(value).strip() == "":
        predicted_set = set()
    else:
        predicted_set = {
            item.strip()
            for item in str(value).split(",")
            if item.strip()
        }

    true_set = truth_lookup.get(
        s1_id,
        set()
    )

    score = f05_score(
        true_set,
        predicted_set
    )

    evaluation_rows.append({
        "source1_entity_id": s1_id,
        "true_matches": len(true_set),
        "predicted_matches": len(predicted_set),
        "f05": score
    })


evaluation = pd.DataFrame(
    evaluation_rows
)


macro_f05 = float(
    evaluation["f05"].mean()
)


pairs_df.to_csv(
    OUTPUT_DIR / "model_features.tsv",
    sep="\t",
    index=False
)


threshold_df.to_csv(
    OUTPUT_DIR / "threshold_results.tsv",
    sep="\t",
    index=False
)


evaluation.to_csv(
    OUTPUT_DIR / "evaluation.tsv",
    sep="\t",
    index=False
)


print("=" * 50)
print("MODEL RESULTS")
print("=" * 50)

print(
    f"Candidate pairs: {len(pairs_df)}"
)

print(
    f"Positive pairs: {int(y.sum())}"
)

print(
    f"Negative pairs: {int((y == 0).sum())}"
)

print(
    f"Best threshold: {BEST_THRESHOLD:.2f}"
)

print(
    f"Macro F0.5: {macro_f05:.4f}"
)

print()

print("=" * 50)
print("MATCHING RESULTS")
print("=" * 50)

display(matching_results)

print()

print("=" * 50)
print("THRESHOLD RESULTS")
print("=" * 50)

display(threshold_df)

print()

print("=" * 50)
print("EVALUATION")
print("=" * 50)

display(evaluation)

MODEL RESULTS
Candidate pairs: 27
Positive pairs: 22
Negative pairs: 5
Best threshold: 0.30
Macro F0.5: 1.0000

MATCHING RESULTS


,source1_entity_id,matched_entity_ids
0,S1_001,"S2_001,S2_009,S3_001,S3_009"
1,S1_002,"S2_002,S2_010,S3_002,S3_010"
2,S1_003,"S2_003,S2_011,S3_003,S3_011"
3,S1_004,"S2_004,S3_004"
4,S1_005,"S2_005,S3_005"
5,S1_006,"S2_006,S3_006"
6,S1_007,"S2_007,S3_007"
7,S1_008,"S2_008,S3_008"



THRESHOLD RESULTS


,threshold,macro_f05
0,0.30,1.000000
1,0.35,1.000000
2,0.40,1.000000
3,0.45,1.000000
4,0.50,1.000000
5,0.55,1.000000
6,0.60,1.000000
7,0.65,1.000000
8,0.70,1.000000
9,0.75,1.000000



EVALUATION


,source1_entity_id,true_matches,predicted_matches,f05
0,S1_001,4,4,1.0
1,S1_002,4,4,1.0
2,S1_003,4,4,1.0
3,S1_004,2,2,1.0
4,S1_005,2,2,1.0
5,S1_006,2,2,1.0
6,S1_007,2,2,1.0
7,S1_008,2,2,1.0
